<img src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcSEaxoapR7GyB6FC7xp4aM-SrjhxTWTdFCmKAX3TvvFvQ&s=10" alt="Tecnológico de Costa Rica" width="160">

# 🌎 WALC 2026
### Ciencia de Datos · Introducción al Aprendizaje Automático
**Día 3 (Miércoles) — Clasificación 2 — Redes neuronales artificiales**

**Instituto Tecnológico de Costa Rica (TEC)** · Instructor: **Efrén Jiménez Delgado**
· 📧 efjimenez@itcr.ac.cr · 🔗 [LinkedIn](https://www.linkedin.com/in/ejimenezdelgado/)

# 🩺 Red neuronal artificial: apoyo al diagnóstico de cáncer de cuello uterino

## 🧵 El hilo conductor (metodología CRISP-DM)

Este cuaderno sigue las seis fases de **CRISP-DM** [1] [2]: cada título de fase marca dónde estamos en el proceso.

| Fase | En este cuaderno |
|---|---|
| 🎯 Comprensión del negocio | Apoyar el diagnóstico de cáncer de cuello uterino a partir de resultados de exámenes y la edad. |
| 🔍 Comprensión de los datos | Cargamos 858 pacientes (UCI), conservamos 9 columnas y revisamos el fuerte desbalance (18 positivos). |
| 🧹 Preparación de los datos | Separamos entrenamiento/prueba (estratificado), balanceamos solo el entrenamiento con CTGAN y estandarizamos. |
| 🤖 Modelado | Entrenamos un perceptrón multicapa (10-5, activación logística) con `scikit-learn`. |
| 📊 Evaluación | Matriz de confusión, ROC/AUC, sensibilidad y especificidad sobre una prueba que no se tocó. |
| 🚀 Despliegue y conclusiones | Interpretamos los resultados con cautela (pocos positivos, variables muy cercanas al diagnóstico). |

> 🎯 **Analogía:** un comité médico que revisa los resultados de varios exámenes (Schiller, citología, biopsia) y vota si hay cáncer; como casi todas las pacientes están sanas, el comité debe aprender a no "dormirse" ante los pocos casos positivos.

### 🗺️ Mapa del curso (cronograma Track 5 · Ciencia de datos)
| Día | Jornada | Teoría y laboratorio |
|---|---|---|
| 1 | Lunes | Predicción: regresión lineal y series de tiempo |
| 2 | Martes | Clasificación 1: árboles de decisión, bosques aleatorios y KNN |
| 👉 **3** | Miércoles | **Clasificación 2: SVM, regresión logística, Naive Bayes y redes neuronales** |
| 4 | Jueves | No supervisado: agrupación y asociación |
| 5 | Viernes | Ciencia de datos con IA generativa, ética y buenas prácticas |

### 📥 Datos del curso
Los archivos se descargan automáticamente (una sola vez) desde el repositorio del curso. Si ya los subió a `sample_data/`, no se vuelven a descargar.

In [ ]:
import os, urllib.request
REPO = 'https://gitlab.com/ejimenez/fiec_intro_machine_learning/-/raw/main'
os.makedirs('sample_data', exist_ok=True)
for nombre, dia in [('kag_risk_factors_cervical_cancer.csv', 'dia_4')]:
    ruta = f'sample_data/{nombre}'
    if not os.path.exists(ruta):
        urllib.request.urlretrieve(f'{REPO}/{dia}/{nombre}', ruta)
        print('Descargado:', ruta)
    else:
        print('Ya existe:', ruta)

---
## 🎯 Fase 1 · Comprensión del negocio

Apoyar el diagnóstico de cáncer de cuello uterino a partir de resultados de exámenes y la edad.

📚 Lectura recomendada: [5] [4]

## Análisis del Problema

Cada año se diagnostican alrededor de 11,000 nuevos casos de cáncer de cuello uterino invasivo en los EE. UU. Sin embargo, la cantidad de nuevos casos de cáncer de cuello uterino ha disminuido de manera constante durante las últimas décadas. Aunque es el tipo de cáncer más prevenible, cada año el cáncer de cuello uterino mata a unas 4.000 mujeres en los EE. UU. Y a unas 300.000 en todo el mundo. En los Estados Unidos, las tasas de mortalidad por cáncer de cuello uterino se redujeron en un 74% entre 1955 y 1992 gracias al aumento de las pruebas de detección y la detección temprana con la prueba de Papanicolaou. EDAD El cincuenta por ciento de los diagnósticos de cáncer de cuello uterino ocurren en mujeres de 35 a 54 años, y aproximadamente el 20% ocurre en mujeres mayores de 65 años. La edad media de diagnóstico es de 48 años. Aproximadamente el 15% de las mujeres desarrollan cáncer de cuello uterino entre los 20 y los 30 años. El cáncer de cuello uterino es extremadamente raro en mujeres menores de 20 años. Sin embargo, muchas mujeres jóvenes se infectan con múltiples tipos de virus del papiloma humano, que luego pueden aumentar su riesgo de contraer cáncer de cuello uterino en el futuro.

> 📌 **Contexto y fuente.** Las cifras epidemiológicas del párrafo anterior provienen de la descripción original del conjunto de datos en Kaggle; para cifras actuales consulte a la OMS [5]. Los datos son de un hospital de Caracas (UCI) [3] [4].

# Clasificar el cancer de cervix

Las redes neuronales artificiales (también conocidas como sistemas conexionistas) son un modelo computacional vagamente inspirado en el comportamiento observado en su homólogo biológico1.
Consiste en un conjunto de unidades, llamadas neuronas artificiales, conectadas entre sí para transmitirse señales. La información de entrada atraviesa la red neuronal (donde se somete a diversas operaciones) produciendo unos valores de salida.

- Red Neuronal

### Autores
   * Efrén Jiménez

### 🧪 Teoría rápida: la red neuronal

Una **neurona artificial** pondera sus entradas, suma un sesgo y aplica una función de activación; varias capas de neuronas forman un **perceptrón multicapa** [8]. Se entrena con propagación hacia adelante y retropropagación del error.

![Perceptrón multicapa](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch12/images/12_03.png)
*Figura: red de tres capas (entrada, oculta, salida). [13]*

![Función sigmoide](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch12/images/12_04.png)
*Figura: activación logística (sigmoide), usada aquí en la capa oculta y en la salida. [13]*

### Librerías

In [ ]:
%pip install -q ctgan
import plotly.graph_objects as go

In [ ]:
import warnings
warnings.filterwarnings('ignore')
#imports
#numpy,pandas,scipy, math, matplotlib
import numpy as np
import pandas as pd
import scipy
from math import sqrt
import matplotlib.pyplot as plt
import seaborn as sns

#model metrics
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from sklearn.model_selection import cross_val_score
from sklearn.metrics import recall_score
from sklearn.metrics import precision_score
from sklearn import metrics

#cross validation
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_curve
from sklearn.metrics import roc_auc_score

import plotly.graph_objects as go

---
## 🔍 Fase 2 · Comprensión de los datos

Cargamos 858 pacientes (UCI), conservamos 9 columnas y revisamos el fuerte desbalance (18 positivos).

📚 Lectura recomendada: [3]

## Entendimiento de los Datos

El total es de 858 registro con 36 columnas dentro del set de datos de entrenamiento

'Age', 'Number of sexual partners', 'First sexual intercourse',
       'Num of pregnancies', 'Smokes', 'Smokes (years)', 'Smokes (packs/year)',
       'Hormonal Contraceptives', 'Hormonal Contraceptives (years)', 'IUD',
       'IUD (years)', 'STDs', 'STDs (number)', 'STDs:condylomatosis',
       'STDs:cervical condylomatosis', 'STDs:vaginal condylomatosis',
       'STDs:vulvo-perineal condylomatosis', 'STDs:syphilis',
       'STDs:pelvic inflammatory disease', 'STDs:genital herpes',
       'STDs:molluscum contagiosum', 'STDs:AIDS', 'STDs:HIV',
       'STDs:Hepatitis B', 'STDs:HPV', 'STDs: Number of diagnosis',
       'STDs: Time since first diagnosis', 'STDs: Time since last diagnosis',
       'Dx:Cancer', 'Dx:CIN', 'Dx:HPV', 'Dx', 'Hinselmann', 'Schiller',
       'Citology', 'Biopsy'

In [ ]:
#Cargar los datos
kag_risk_factors_cervical_cancer = pd.read_csv(r"sample_data/kag_risk_factors_cervical_cancer.csv",na_values='?')

In [ ]:
kag_risk_factors_cervical_cancer.columns

In [ ]:
kag_risk_factors_cervical_cancer=kag_risk_factors_cervical_cancer.drop(
    columns=["Number of sexual partners",
             "STDs: Time since first diagnosis",
             "STDs: Number of diagnosis",
             "STDs: Time since last diagnosis",
             "First sexual intercourse",
             "Num of pregnancies",
             "Smokes",
             "Smokes (years)",
             "Smokes (packs/year)",
             "Hormonal Contraceptives",
             "Hormonal Contraceptives (years)",
             "IUD",
             "IUD (years)",
             "STDs",
             "STDs (number)",
             "STDs:condylomatosis",
             "STDs:cervical condylomatosis",
             "STDs:vaginal condylomatosis",
             "STDs:vulvo-perineal condylomatosis",
             "STDs:syphilis",
             "STDs:pelvic inflammatory disease",
             "STDs:genital herpes",
             "STDs:molluscum contagiosum",
             "STDs:AIDS",
             "STDs:HIV",
             "STDs:Hepatitis B",
             "STDs:HPV"
            ])

In [ ]:
#Primeros registros
kag_risk_factors_cervical_cancer.head()

In [ ]:
#Describir dataset
kag_risk_factors_cervical_cancer.describe()

In [ ]:
#Información del dataset
kag_risk_factors_cervical_cancer.info()

In [ ]:
# Numero de registros del dataset
print("Cantidad de registros:",len(kag_risk_factors_cervical_cancer))
# Cantidad de columnas del dataset
print("Cantidad de columnas:",len(kag_risk_factors_cervical_cancer.columns))
# Tipos de datos del dataset
print("Tipo de datos:",kag_risk_factors_cervical_cancer.dtypes.unique())
# Tamaño del daset
print("Filas y columnas:",kag_risk_factors_cervical_cancer.shape)

In [ ]:
# Chekear valores nulos
print(kag_risk_factors_cervical_cancer.isnull().any().sum(), ' / ', len(kag_risk_factors_cervical_cancer.columns))

In [ ]:
#Cuales columnnas estas con valores nulos
kag_risk_factors_cervical_cancer.isnull().sum()[kag_risk_factors_cervical_cancer.isnull().sum() > 0]

In [ ]:
# N/A= Valores vacios
#Usar solo si se quiere eliminar los registros que contienen columnas vacías
kag_risk_factors_cervical_cancer=kag_risk_factors_cervical_cancer.dropna()

In [ ]:
kag_risk_factors_cervical_cancer

In [ ]:
kag_risk_factors_cervical_cancer["Dx:Cancer"].unique()

In [ ]:
kag_risk_factors_cervical_cancer['Dx:Cancer'].value_counts(normalize=True)

In [ ]:
import plotly.express as px
import plotly.figure_factory as ff
fig = px.histogram(kag_risk_factors_cervical_cancer, x="Dx:Cancer", nbins=2, color="Dx:Cancer")
fig.update_layout(title_text="Distribución del rango de cáncer de cérvix en el Dataset")
fig.show()

---
## 🔍 Fase 2 · Comprensión de los datos — exploración de los datos

Cargamos 858 pacientes (UCI), conservamos 9 columnas y revisamos el fuerte desbalance (18 positivos).

## Exploración de los Datos

#### Gráficos exploratorios

In [ ]:
plt.figure(1, figsize=(15, 6))
n = 0
for x in ['Schiller', 'Biopsy', 'Hinselmann']:
    n += 1
    plt.subplot(1, 3, n)
    sns.histplot(kag_risk_factors_cervical_cancer[x], bins=20)
    plt.title('Distribución de {}'.format(x))
plt.tight_layout()
plt.show()

In [ ]:
kag_risk_factors_cervical_cancer.sort_values(by=['Age'])["Age"].value_counts().index

In [ ]:
df = kag_risk_factors_cervical_cancer.groupby(by=["Age"])['Age'].size().nlargest(8).reset_index(name='top8')

In [ ]:
labels = df.Age.value_counts().index
colors = ['green','blue','red','yellow','orange','gray','lime','pink']
explode = [0,0,0,0,0,0,0,0,]
sizes = df.Age.value_counts().values

# visual
plt.figure(figsize = (7,7))
plt.pie(sizes, explode=explode, labels=labels, colors=colors, autopct='%1.1f%%')
plt.title('Distribucion de la edad de las mujeres',color = 'blue',fontsize = 15)

In [ ]:
kag_risk_factors_cervical_cancer

In [ ]:
sns.boxplot(x="Dx:Cancer", y="Age", hue="Dx:Cancer", data=kag_risk_factors_cervical_cancer, palette="PRGn")
plt.show()

### ⚖️ Separar primero, balancear después

Con **18 positivos de 858**, el modelo tendería a decir "sano" siempre. Una solución es **generar casos positivos sintéticos** con una *GAN condicional para datos tabulares* (**CTGAN**) [6]. ⚠️ **Se hace después de separar** entrenamiento y prueba y **solo con el entrenamiento**: si se generaran antes, los casos sintéticos (copias aproximadas de pacientes reales) caerían también en la prueba y la evaluación saldría artificialmente optimista.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

columnas = ["Age", "Dx:CIN", "Dx:HPV", "Dx", "Hinselmann", "Schiller", "Citology", "Biopsy"]
X_all = kag_risk_factors_cervical_cancer[columnas]
y_all = kag_risk_factors_cervical_cancer["Dx:Cancer"]

# 1) separar (estratificado: conserva la proporción de positivos en ambas partes)
X_train, X_test, y_train, y_test = train_test_split(X_all, y_all, test_size=0.3, random_state=1234, stratify=y_all)
print('Positivos en entrenamiento:', int(y_train.sum()), '| en prueba:', int(y_test.sum()))

# 2) generar positivos sintéticos con CTGAN usando SOLO el entrenamiento
import numpy as np, torch
from ctgan import CTGAN
np.random.seed(0); torch.manual_seed(0)
positivos = X_train.assign(**{'Dx:Cancer': y_train})[y_train == 1]
faltan = int((y_train == 0).sum() - (y_train == 1).sum())
ctgan = CTGAN(epochs=100, pac=1, verbose=False)      # pac=1 porque hay muy pocos positivos
ctgan.fit(positivos, discrete_columns=list(positivos.columns))
samples = ctgan.sample(faltan)
X_train = pd.concat([X_train, samples[columnas]], ignore_index=True)
y_train = pd.concat([y_train, samples['Dx:Cancer']], ignore_index=True)
print('Clases en el entrenamiento balanceado:'); print(y_train.value_counts())

# 3) estandarizar con la media/desviación del entrenamiento
sc = StandardScaler().fit(X_train)
X_train = pd.DataFrame(sc.transform(X_train), columns=columnas)
X_test = pd.DataFrame(sc.transform(X_test), columns=columnas, index=X_test.index)

>Es importante siempre validar los rangos de los conjuntos de datos creados, para evitar caer en extrapolación:

In [ ]:
print("Entrenamiento dimensiones",X_train.shape)
print("Descripción")
X_train.describe()

In [ ]:
print("Pruebas dimensiones",X_test.shape)
print("Descripción")
X_test.describe()

---
## 🤖 Fase 4 · Modelado

Entrenamos un perceptrón multicapa (10-5, activación logística) con `scikit-learn`.

📚 Lectura recomendada: [8] [9]

## Modelo de Machine Learning

Una vez seleccionadas las variables para incluir en el modelo de regresión, se procede a crearlo:

In [ ]:
#--------------
# ANN
#--------------
from sklearn.neural_network import MLPClassifier
#model = MLPClassifier(activation='tanh', max_iter=10000, hidden_layer_sizes=(18,9,4,3))

model = MLPClassifier(hidden_layer_sizes=(10,5), max_iter=300,
                    solver='adam', verbose=False, random_state=1,activation='logistic',
                    learning_rate_init=0.01)

In [ ]:
# Entrenamos nuestro modelo
model.fit(X_train, y_train)

In [ ]:
# Obtener el historial de entrenamiento del modelo
history = model.loss_curve_
# Crear figura con el gráfico de loss
fig = go.Figure()
fig.add_trace(go.Scatter(x=list(range(len(history))),
                         y=history,
                         mode='lines',
                         name='Loss'))

fig.update_layout(title='Gráfico de Loss',
                  xaxis_title='Iteraciones',
                  yaxis_title='Loss')

# Mostrar figura
fig.show()

In [ ]:
# Arquitectura de la red entrenada: forma de la matriz de pesos entre capas consecutivas
for i, w in enumerate(model.coefs_):
    print(f"Capa {i} → capa {i+1}: matriz de pesos {w.shape[0]} × {w.shape[1]}")

In [ ]:
# Hacemos las predicciones que en definitiva una línea (en este caso, al ser 2D)
y_pred1 = model.predict( X_test )

In [ ]:
y_pred1

In [ ]:
predict_values=model.predict_proba(X_test)
predict_values[:, 1]

---
## 📊 Fase 5 · Evaluación

Matriz de confusión, ROC/AUC, sensibilidad y especificidad sobre una prueba que no se tocó.

📚 Lectura recomendada: [11] [12]

# Evaluación


In [ ]:
np.array(y_pred1)

In [ ]:
matrix=confusion_matrix(np.array(y_test), np.array(y_pred1))
matrix

In [ ]:
figure = plt.figure(figsize=(8, 8))
sns.heatmap(matrix, annot=True, fmt='g', cmap=plt.cm.Blues)
plt.tight_layout()
plt.ylabel('Real')
plt.xlabel('Predicho')
plt.show()

In [ ]:
# Validación cruzada de 5 particiones solo sobre el entrenamiento
scores = cross_val_score(model, X_train, y_train, cv=5)
print("Exactitud media en validación cruzada (entrenamiento): %.2f, desv. estándar = %.2f" % (scores.mean(), scores.std()))
print("Exactitud en el entrenamiento:", round(model.score(X_train, y_train), 3))
print("Exactitud en la prueba (datos reales no vistos):", round(model.score(X_test, y_test), 3))

In [ ]:
auc = roc_auc_score(y_test, predict_values[:, 1])
print('AUC: %.2f' % auc)

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, predict_values[:, 1])
plt.plot(fpr, tpr, color='orange', label='ROC')

plt.plot([0, 1], [0, 1], color='darkblue', linestyle='--')

plt.xlabel('Tasa de falsos positivos')
plt.ylabel('Tasa de verdaderos positivos')
plt.title('Curva ROC')
plt.legend()
plt.show()

\begin{equation}Exactitud=\frac{VP+VN}{Total}\end{equation}

In [ ]:
VN, FP, FN, VP = matrix.ravel()
(VP + VN) / matrix.sum()

\begin{equation}\text{Sensibilidad}=\frac{VP}{\text{Total Positivos}}\end{equation}

In [ ]:
VP / (VP + FN)   # sensibilidad: de las pacientes con cáncer, ¿cuántas detecta el modelo?

\begin{equation}\text{Especificidad}=\frac{VN}{\text{Total Negativos}}\end{equation}

In [ ]:
VN / (VN + FP)   # especificidad: de las pacientes sanas, ¿cuántas clasifica bien?

---
## 🚀 Fase 6 · Despliegue y conclusiones

Interpretamos los resultados con cautela (pocos positivos, variables muy cercanas al diagnóstico).

📚 Lectura recomendada: [12]

## Resultados

En la ejecución de referencia, sobre la prueba real (258 pacientes, 5 con cáncer) la red detecta **las 5 positivas** (sensibilidad 100 %) a costa de **9 falsos positivos** (especificidad ≈ 96.4 %, exactitud ≈ 96.5 %), con un AUC cercano a 1. Con tan pocos positivos esas cifras son muy inestables, y son tan buenas porque las variables de entrada (`Dx`, `Dx:CIN`, `Dx:HPV`, `Biopsy`, `Schiller`…) son resultados de exámenes casi equivalentes al diagnóstico final. En un tamizaje real un falso negativo es más grave que un falso positivo, por lo que se prefiere un modelo de alta sensibilidad.

---
## 🧪 Refuerzo: evaluar bien cuando casi todos los casos son negativos (clases desbalanceadas)

En este conjunto solo **18 de 858 pacientes (2.1 %)** tienen `Dx:Cancer = 1`. Un modelo que diga siempre "no tiene cáncer" tendría 97.9 % de exactitud y sería inútil: por eso se mira la **sensibilidad** (*recall*), la **precisión** y el **AUC**, no solo la exactitud [12]. Con tan pocos positivos, una sola partición entrenamiento/prueba es muy inestable (5 positivos en prueba); se usa **validación cruzada estratificada repetida**. Además, el balanceo (aquí, sobremuestreo aleatorio de los positivos) se hace **solo dentro del entrenamiento de cada partición**, nunca antes de separar, para no filtrar información a la prueba [7].

In [ ]:
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import recall_score, precision_score, roc_auc_score
from sklearn.utils import resample

cols = ["Age", "Dx:CIN", "Dx:HPV", "Dx", "Hinselmann", "Schiller", "Citology", "Biopsy"]
Xc = kag_risk_factors_cervical_cancer[cols].reset_index(drop=True)
yc = kag_risk_factors_cervical_cancer["Dx:Cancer"].reset_index(drop=True)
cv = RepeatedStratifiedKFold(n_splits=3, n_repeats=5, random_state=0)   # 18 positivos → 6 por partición de prueba

def evaluar(balancear):
    filas = []
    for tr, te in cv.split(Xc, yc):
        Xtr, ytr = Xc.iloc[tr], yc.iloc[tr]
        if balancear:      # sobremuestreo aleatorio de los positivos, SOLO en el entrenamiento
            pos, neg = Xtr[ytr == 1], Xtr[ytr == 0]
            pos_up = resample(pos, replace=True, n_samples=len(neg), random_state=0)
            Xtr = pd.concat([neg, pos_up]); ytr = pd.Series([0] * len(neg) + [1] * len(pos_up))
        modelo = make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(10, 5), activation='logistic', max_iter=500, random_state=1))
        modelo.fit(Xtr, ytr)
        prob = modelo.predict_proba(Xc.iloc[te])[:, 1]; pred = (prob >= 0.5).astype(int)
        filas.append({'sensibilidad': recall_score(yc.iloc[te], pred), 'precisión': precision_score(yc.iloc[te], pred, zero_division=0), 'AUC': roc_auc_score(yc.iloc[te], prob)})
    return pd.DataFrame(filas).mean().round(3)

pd.DataFrame({'sin balancear': evaluar(False), 'sobremuestreo aleatorio': evaluar(True)})

### ✍️ Ejercicios
1. ¿Qué estrategia detecta más casos positivos (sensibilidad)? ¿A costa de qué métrica?
2. En tamizaje de cáncer, ¿es peor un falso negativo o un falso positivo? ¿Qué umbral de probabilidad usaría en lugar de 0.5?
3. Las variables `Hinselmann`, `Schiller`, `Citology` y `Biopsy` son **resultados de exámenes de diagnóstico**, no factores de riesgo. ¿Qué significa que el modelo las use para predecir `Dx:Cancer`?

---
## ✅ Conclusiones y siguientes pasos

Con solo 18 pacientes positivas, las métricas de una única partición son muy inestables; lo prudente es repetir la validación cruzada y reportar rangos. Además, el modelo usa **resultados de exámenes** (citología, Schiller, biopsia) que están casi tan cerca del diagnóstico como la etiqueta misma; sirve como apoyo a la decisión, **no como herramienta de pronóstico de riesgo**. La versión original generaba casos sintéticos antes de separar los datos y contaminaba la prueba; aquí se corrigió [6].

---
## 📚 Referencias

Fuentes adicionales para profundizar: [10].

**[1]** Chapman, P., Clinton, J., Kerber, R., Khabaza, T., Reinartz, T., Shearer, C., & Wirth, R. (2000). *CRISP-DM 1.0: Step-by-step data mining guide*. SPSS Inc. https://www.the-modeling-agency.com/crisp-dm.pdf

**[2]** Wirth, R., & Hipp, J. (2000). CRISP-DM: Towards a standard process model for data mining. *Proceedings of the 4th International Conference on the Practical Applications of Knowledge Discovery and Data Mining*, 29–39.

**[3]** Fernandes, K., Cardoso, J. S., & Fernandes, J. (2017). *Cervical cancer (Risk Factors)* [Conjunto de datos]. UCI Machine Learning Repository. https://archive.ics.uci.edu/dataset/383/cervical+cancer+risk+factors

**[4]** Fernandes, K., Cardoso, J. S., & Fernandes, J. (2017). Transfer learning with partial observability applied to cervical cancer screening. En *Pattern Recognition and Image Analysis (IbPRIA 2017), LNCS 10255* (pp. 243–250). Springer. https://doi.org/10.1007/978-3-319-58838-4_27

**[5]** Organización Mundial de la Salud. (2020). *Estrategia mundial para acelerar la eliminación del cáncer del cuello uterino como problema de salud pública*. OMS. https://www.who.int/publications/i/item/9789240014107

**[6]** Xu, L., Skoularidou, M., Cuesta-Infante, A., & Veeramachaneni, K. (2019). Modeling tabular data using conditional GAN. *Advances in Neural Information Processing Systems, 32*. https://arxiv.org/abs/1907.00503

**[7]** Chawla, N. V., Bowyer, K. W., Hall, L. O., & Kegelmeyer, W. P. (2002). SMOTE: Synthetic minority over-sampling technique. *Journal of Artificial Intelligence Research, 16*, 321–357. https://doi.org/10.1613/jair.953

**[8]** Rumelhart, D. E., Hinton, G. E., & Williams, R. J. (1986). Learning representations by back-propagating errors. *Nature, 323*(6088), 533–536. https://doi.org/10.1038/323533a0

**[9]** Goodfellow, I., Bengio, Y., & Courville, A. (2016). *Deep Learning*. MIT Press. https://www.deeplearningbook.org

**[10]** Pedregosa, F., Varoquaux, G., Gramfort, A., Michel, V., Thirion, B., Grisel, O., … Duchesnay, E. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research, 12*, 2825–2830. https://jmlr.org/papers/v12/pedregosa11a.html

**[11]** Fawcett, T. (2006). An introduction to ROC analysis. *Pattern Recognition Letters, 27*(8), 861–874. https://doi.org/10.1016/j.patrec.2005.10.010

**[12]** Powers, D. M. W. (2011). Evaluation: From precision, recall and F-measure to ROC, informedness, markedness and correlation. *Journal of Machine Learning Technologies, 2*(1), 37–63.

**[13]** Raschka, S., & Mirjalili, V. (2019). *Python Machine Learning* (3.ª ed.). Packt Publishing. (Figuras usadas desde https://github.com/rasbt/python-machine-learning-book-3rd-edition)